# Kort sammanfattning – ML-lektion

## 1. Solvers

### Solver

`solver` anger vilken algoritm som används för att optimera modellen och beräkna dess parametrar under träningen. När en maskininlärningsmodell tränas måste den lösa ett matematiskt optimeringsproblem för att hitta de bästa vikterna eller koefficienterna som passar datan.

Olika **solvers** använder olika matematiska metoder för att lösa detta problem. Valet av solver kan påverka modellens **hastighet, minnesanvändning och stabilitet**, särskilt beroende på hur stort datasetet är.

`solver` används i flera modeller i scikit-learn, till exempel i **Ridge Regression, Logistic Regression och neurala nätverk**.

### Exempel på solvers

Några vanliga `solver` i scikit-learn är:

- **`lbfgs`** – en effektiv optimeringsalgoritm som ofta används som standard, särskilt i Logistic Regression.
- **`sag`** – Stochastic Average Gradient, snabb för stora dataset.
- **`saga`** – en förbättrad version av SAG som stödjer både L1- och L2-regularisering.
- **`cholesky`** – använder Cholesky-dekomposition för att lösa linjära ekvationer, vanligt i Ridge Regression.
- **`svd`** – använder singular value decomposition och är stabil för mindre dataset.

- **SAG** och **SAGA** är stochastic-metoder.
- **SAGA** är oftast bättre i praktiken eftersom den stöder:
  - **L1**
  - **L2**
  - **Elastic Net**
- Idén: välj solver efter problem och regularisering.

## 2. Polynomial Regression
- Vanlig linjär regression räcker inte om sambandet är böjt.
- Då använder man **polynomial features**.
- **Degree 1** = för enkel modell → risk för **underfitting**.
- **Hög degree** = för komplex modell → risk för **overfitting**.
- poäng: modellen ska fånga mönstret, inte memorera punkterna.

## 3. Learning Curves
- Learning curves visar:
  - **train error**
  - **validation error**
  - mot **träningsmängd**
- Bra modell:
  - kurvorna blir **nära varandra**
  - validation error stabiliseras
- Overfitting:
  - låg train error
  - högre validation error
- Underfitting:
  - båda är höga

## 4. Regularisering

Regularisering används för att minska överanpassning (overfitting) genom att begränsa hur stora modellens parametrar får bli.  
Det gör modellen mer stabil och minskar variansen.

- **L2 (Ridge)** försöker hålla parametrarna små genom att lägga till ett straff baserat på kvadraten av vikterna. Vikterna blir mindre men sällan exakt noll.
- **L1 (Lasso)** använder absolutbelopp av vikterna och kan sätta vissa parametrar exakt till noll. Detta gör att modellen kan välja bort vissa features.
- Regularisering påverkar optimeringen genom att lägga till en extra term i kostnadsfunktionen, vilket gör att gradient descent både minskar felet och samtidigt trycker vikterna mot noll.

### L2 / Ridge
- Gör modellen mjukare
- Håller parametrarna små
- Minskar varians

### L1 / Lasso
- Mer aggressiv
- Kan sätta vissa parametrar exakt till **0**
- Bra för feature selection

### Elastic Net
- Kombination av L1 och L2

## 5. Early Stopping

### Early Stopping

Early stopping är en teknik för att förhindra **overfitting** genom att stoppa träningen av en modell vid rätt tidpunkt.

Under träningen minskar vanligtvis **training error** hela tiden eftersom modellen blir bättre på att passa träningsdatan. Samtidigt följer man **validation error**, som visar hur bra modellen fungerar på data den inte tränats på.

I början minskar både training error och validation error. Efter ett tag börjar dock validation error öka igen. Detta betyder att modellen börjar **överanpassa träningsdatan** och får sämre generalisering.

Early stopping fungerar genom att:
- mäta validation error efter varje epok
- spara den modell som ger **lägst validation error**
- stoppa träningen när validation error inte längre förbättras

Ofta används en liten **tolerans (patience)** så att träningen inte stoppas direkt om felet bara varierar lite mellan epoker.

Fördelar:
- minskar overfitting
- ger bättre generalisering på ny data
- sparar beräkningstid

Kort sagt: Early stopping väljer den modell som presterar bäst på validation data och använder den för att generalisera till ny data.

- Man tränar inte bara ett fast antal epoker.
- Man tittar på validation error.
- När validation error börjar bli sämre bör man stoppa.
- Poängen: undvika overfitting.

## 6. Logistic Regression

Logistisk regression används för **klassificering**, alltså för att avgöra vilken klass en datapunkt tillhör. Till skillnad från linjär regression förutsäger modellen inte ett kontinuerligt värde utan en **sannolikhet**.

Grunden i logistisk regression är **sigmoidfunktionen**, som omvandlar ett värde från intervallet \(-\infty, +\infty\) till ett värde mellan **0 och 1**. Detta värde tolkas som sannolikheten att datapunkten tillhör en viss klass.

\[
\sigma(x) = \frac{1}{1 + e^{-x}}
\]

För att bestämma klassen används ofta en **decision boundary**.  
Om sannolikheten är större än ett tröskelvärde (t.ex. 0.5) klassificeras datapunkten som den positiva klassen, annars som den negativa klassen.

## 8. Multiclass och Softmax

När problemet har fler än två klasser används en utökning som kallas **Softmax Regression**, där modellen beräknar sannolikheter för alla klasser och väljer den klass med högst sannolikhet.


## 9. Viktig huvudidé från lektionen
Läraren försöker visa hur dessa saker hänger ihop:
- modellens komplexitet
- overfitting / underfitting
- regularisering
- learning curves
- val av solver
- klassificering med logistic regression

---

# Kort minneslista
- **Degree för låg** → underfitting
- **Degree för hög** → overfitting
- **L2** → mindre vikter
- **L1** → vissa vikter blir 0
- **Learning curves** → hjälper att se modellproblem
- **Early stopping** → stoppa när validation blir sämre
- **Sigmoid** → sannolikhet mellan 0 och 1
- **Softmax** → flera klasser


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.model_selection import learning_curve
from sklearn.datasets import make_regression, load_iris

# -----------------------------
# 1. Polynomial Regression
# -----------------------------
np.random.seed(42)
X = 6 * np.random.rand(100, 1) - 3
y = 0.5 * X[:, 0]**2 + X[:, 0] + 2 + np.random.randn(100)

poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linreg", LinearRegression())
])
poly_model.fit(X, y)

# -----------------------------
# 2. Learning Curve
# -----------------------------
train_sizes, train_scores, valid_scores = learning_curve(
    LinearRegression(), X, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring="neg_root_mean_squared_error"
)

train_errors = -train_scores.mean(axis=1)
valid_errors = -valid_scores.mean(axis=1)

plt.figure(figsize=(6, 4))
plt.plot(train_sizes, train_errors, "r-+", linewidth=2, label="train")
plt.plot(train_sizes, valid_errors, "b-", linewidth=2, label="valid")
plt.xlabel("Training set size")
plt.ylabel("RMSE")
plt.title("Learning Curve")
plt.grid()
plt.legend()
plt.show()

# -----------------------------
# 3. Ridge (L2 regularisering)
# -----------------------------
ridge_model = Pipeline([
    ("poly", PolynomialFeatures(degree=10, include_bias=False)),
    ("ridge", Ridge(alpha=0.1))
])
ridge_model.fit(X, y)

# -----------------------------
# 4. Logistic Regression
# -----------------------------
iris = load_iris()
X_iris = iris.data[:, [2]]  # petal length
y_iris = (iris.target == 2).astype(int)  # 1 om Virginica, annars 0

log_reg = LogisticRegression()
log_reg.fit(X_iris, y_iris)

print("Klar:")
print("- Polynomial Regression")
print("- Learning Curve")
print("- Ridge (L2)")
print("- Logistic Regression")
